# Classify all beach cells with the globally weighted ensemble

This notebook reads complete cells from `Data/Raw/beach_all`, loads the four best saved models, and combines their probabilities using the per-model weights calculated from validation macro-F1. Each model uses its own saved context radius, normalization statistics, and image size.

Results are written to `Results/bs_all_cells_classification/weighted_ensemble/<setup>/<date>/classification.csv`. No input matrix files are copied.

## Channel-ablation setup
Select `RGB_only`, `RGB_slope`, `RGB_elevation`, or `RGB_slope_elevation` with `BEACH_CHANNEL_SETUP` (default: `RGB_slope_elevation`). Training and ensemble evaluation are independent for each setup, retaining identical splits/CV plans, parameter grids, and seeds. Only selected channels are read and normalized; the model input-channel count and saved metadata match the selection. Results are stored in the corresponding setup subfolder. Existing outputs below are retained and may predate this change; rerun to obtain setup-specific results.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.channel_setups import get_channel_setup, setup_result_dir, validate_setup_metadata
from Source.ensemble import (
    classify_date_directories_with_global_ensemble,
    load_global_weighted_ensemble,
    load_model_artifact,
    validate_artifact_compatibility,
)
from Source.inference import discover_date_directories
from Source.training import get_device

In [ ]:
# Select a setup here, or export BEACH_CHANNEL_SETUP before batch execution.
CHANNEL_SETUP, FEATURE_SUFFIXES = get_channel_setup()
print(f"Channel setup: {CHANNEL_SETUP}; features: {FEATURE_SUFFIXES}", flush=True)

SOURCE_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
ENSEMBLE_RESULTS_DIR = setup_result_dir(PROJECT_ROOT / "Results" / "bs_ensemble", CHANNEL_SETUP)
ENSEMBLE_DEFINITION_PATH = ENSEMBLE_RESULTS_DIR / "ensemble_definition.json"
OUTPUT_ROOT = setup_result_dir(PROJECT_ROOT / "Results" / "bs_all_cells_classification" / "weighted_ensemble", CHANNEL_SETUP)

MODEL_RESULT_DIRS = {
    "convnext_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_convnext_tiny", CHANNEL_SETUP),
    "vit_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_vit_tiny", CHANNEL_SETUP),
    "swin_tiny": setup_result_dir(PROJECT_ROOT / "Results" / "bs_swin_tiny", CHANNEL_SETUP),
    "efficientnetv2_s": setup_result_dir(PROJECT_ROOT / "Results" / "bs_efficientnetv2_s", CHANNEL_SETUP),
}

BATCH_SIZE = 64
NUM_WORKERS = 0
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}", flush=True)
print(f"Source root:        {SOURCE_ROOT}", flush=True)
print(f"Ensemble definition: {ENSEMBLE_DEFINITION_PATH}", flush=True)
print(f"Output root:        {OUTPUT_ROOT}", flush=True)

In [ ]:
print("Checking inputs and model artifacts...", flush=True)
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Cell data directory not found: {SOURCE_ROOT}")
if not ENSEMBLE_DEFINITION_PATH.exists():
    raise FileNotFoundError(
        f"Ensemble definition not found: {ENSEMBLE_DEFINITION_PATH}. "
        "Run bs_evaluate_ensemble.ipynb first."
    )

artifacts = [
    load_model_artifact(name, result_dir)
    for name, result_dir in MODEL_RESULT_DIRS.items()
]
validate_artifact_compatibility(artifacts)
for artifact in artifacts:
    validate_setup_metadata(artifact.metadata, CHANNEL_SETUP)
reference_metadata = artifacts[0].metadata
idx_to_class_code = {
    int(index): str(code)
    for index, code in reference_metadata["idx_to_class_code"].items()
}

artifact_table = pd.DataFrame([
    {
        "model": artifact.name,
        "context_radius": int(artifact.metadata["context_radius"]),
        "image_size": int(artifact.metadata["image_size"]),
        "checkpoint": str(artifact.checkpoint_path),
    }
    for artifact in artifacts
])
artifact_table

In [ ]:
print("Selecting compute device and loading all four best models...", flush=True)
device = get_device()
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(device)}", flush=True)

models, model_weights, ensemble_definition = load_global_weighted_ensemble(
    artifacts,
    ENSEMBLE_DEFINITION_PATH,
    device,
)
print("Loaded global model weights:", flush=True)
for model_name, weight in model_weights.items():
    print(f"  {model_name}: {weight:.6f}", flush=True)
print(
    f"Validation-selected method recorded in definition: "
    f"{ensemble_definition.get('validation_selected_method')}",
    flush=True,
)

In [ ]:
print("Discovering date folders...", flush=True)
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}", flush=True)
print(
    "Each date is scanned once for complete target cells. "
    "Neighbor files are then read on demand using each model's context radius.",
    flush=True,
)

In [ ]:
print("Starting weighted-ensemble classification...", flush=True)
classification_summary = classify_date_directories_with_global_ensemble(
    date_dirs=date_dirs,
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    artifacts=artifacts,
    models=models,
    weights=model_weights,
    idx_to_class_code=idx_to_class_code,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    base_suffix=BASE_SUFFIX,
    device=device,
)
print("Weighted-ensemble classification finished.", flush=True)
classification_summary